what  corelation tell  us  

+1 → stocks move together
0 → no relation
-1 → move opposite

Data Loading & Cleaning

Reads CSV files for multiple stocks
Cleans columns (removes commas, fixes date format)
Converts prices and volume into numeric values


Trend Indicators
SMA (Simple Moving Average)
EMA (Exponential Moving Average)


Trading Signals
Signal = EMA5 > EMA10
Crossover = Signal.diff()

BUY → when EMA5 crosses above EMA10

SELL → when EMA5 crosses below EMA10


Returns & Volatility
Return = pct_change()
Volatility = rolling std

Measures daily price change
Measures risk using standard deviation


Align All Stocks
price_df = joined on Date



Compute Correlation
corr_matrix = price_df.corr()

| Value | Meaning       |
| ----- | ------------- |
| +1    | Move together |
| 0     | No relation   |
| -1    | Move opposite |



The script generates:
correlation_matrix.csv
correlation_heatmap.png
corelation_output.csv

output_charts_data_analysisoutput will generate in output_charts_corelation this  folder



In [1]:
"""
Stock Analysis Script (FULL: SMA + EMA + Signals + Volatility + Correlation)
"""

import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import os
from pathlib import Path
import itertools

# ─────────────────────────────────────────────
# CONFIG
# ─────────────────────────────────────────────
CSV_FILES = {
    "HLL":      r"E:\KEC TASK\Download data\HLL Historical Data.csv",
    "TCS":      r"E:\KEC TASK\Download data\TCS Historical Data.csv",
    "RELIANCE": r"E:\KEC TASK\Download data\RELI Historical Data.csv",
    "MSFT":     r"E:\KEC TASK\Download data\MSFT Historical Data.csv",
    "HDBK":     r"E:\KEC TASK\Download data\HDBK Historical Data (1).csv",
}

OUTPUT_DIR = "output_charts_corelation"
os.makedirs(OUTPUT_DIR, exist_ok=True)

colors = itertools.cycle(["#2196F3", "#FF5722", "#4CAF50", "#9C27B0", "#FF9800"])

# ─────────────────────────────────────────────
# HELPERS
# ─────────────────────────────────────────────
def parse_volume(vol_str):
    vol_str = str(vol_str).strip().upper().replace(",", "")
    if vol_str in ("", "N/A", "NAN", "-"):
        return np.nan
    multipliers = {"K": 1e3, "M": 1e6, "B": 1e9}
    for s, m in multipliers.items():
        if vol_str.endswith(s):
            return float(vol_str[:-1]) * m
    return float(vol_str)


def load_stock(filepath, ticker):
    if not Path(filepath).exists():
        print(f"[SKIP] {ticker}")
        return None

    df = pd.read_csv(filepath)
    df.columns = [c.strip().strip('"') for c in df.columns]

    df["Date"] = pd.to_datetime(df["Date"], dayfirst=True, errors="coerce")
    df = df.dropna(subset=["Date"]).sort_values("Date")

    for col in ["Price", "Open", "High", "Low"]:
        if col in df.columns:
            df[col] = df[col].astype(str).str.replace(",", "").astype(float)

    if "Vol." in df.columns:
        df["Volume"] = df["Vol."].apply(parse_volume)

    df["Ticker"] = ticker
    return df


# ─────────────────────────────────────────────
# LOAD DATA
# ─────────────────────────────────────────────
stocks = {}
for t, p in CSV_FILES.items():
    df = load_stock(p, t)
    if df is not None:
        stocks[t] = df

# ─────────────────────────────────────────────
# INDICATORS
# ─────────────────────────────────────────────
for t, df in stocks.items():

    # SMA
    df["SMA10"] = df["Price"].rolling(10).mean()

    # EMA
    df["EMA5"] = df["Price"].ewm(span=5).mean()
    df["EMA10"] = df["Price"].ewm(span=10).mean()

    # Signals
    df["Signal"] = np.where(df["EMA5"] > df["EMA10"], 1, 0)
    df["Crossover"] = df["Signal"].diff()

    # Returns & Volatility
    df["Return"] = df["Price"].pct_change()
    df["Volatility"] = df["Return"].rolling(10).std()


# ─────────────────────────────────────────────
# 🔥 CORRELATION ANALYSIS
# ─────────────────────────────────────────────

# Step 1: Create aligned price dataframe
price_df = pd.DataFrame()

for t, df in stocks.items():
    temp = df[["Date", "Price"]].rename(columns={"Price": t})
    temp = temp.set_index("Date")
    price_df = price_df.join(temp, how="outer") if not price_df.empty else temp

price_df = price_df.sort_index().fillna(method="ffill")

# Step 2: Compute correlation
corr_matrix = price_df.corr()

# Save CSV
corr_matrix.to_csv(os.path.join(OUTPUT_DIR, "correlation_matrix.csv"))

print("\n📊 Correlation Matrix:\n")
print(corr_matrix)

# ─────────────────────────────────────────────
# 🔥 CORRELATION HEATMAP
# ─────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(8, 6))

cax = ax.matshow(corr_matrix, cmap="coolwarm")
plt.colorbar(cax)

ax.set_xticks(range(len(corr_matrix.columns)))
ax.set_yticks(range(len(corr_matrix.columns)))

ax.set_xticklabels(corr_matrix.columns, rotation=45)
ax.set_yticklabels(corr_matrix.columns)

# Annotate values
for i in range(len(corr_matrix.columns)):
    for j in range(len(corr_matrix.columns)):
        ax.text(j, i, f"{corr_matrix.iloc[i, j]:.2f}",
                va="center", ha="center", color="black")

plt.title("Stock Correlation Heatmap")
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "correlation_heatmap.png"))
plt.close()


# ─────────────────────────────────────────────
# SAVE FINAL DATA
# ─────────────────────────────────────────────
all_data = pd.concat(stocks.values())
all_data.to_csv(os.path.join(OUTPUT_DIR, "corelation_output.csv"), index=False)

print("\n✅ DONE — Correlation + Full Analysis Ready!")

C:\Users\skitd\AppData\Local\Temp\ipykernel_27360\4280594994.py:106: FutureWarning: DataFrame.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  price_df = price_df.sort_index().fillna(method="ffill")



📊 Correlation Matrix:

               HLL       TCS  RELIANCE      MSFT      HDBK
HLL       1.000000  0.258882  0.459767  0.874891  0.482920
TCS       0.258882  1.000000 -0.048589  0.309512  0.792353
RELIANCE  0.459767 -0.048589  1.000000  0.497344 -0.008441
MSFT      0.874891  0.309512  0.497344  1.000000  0.481077
HDBK      0.482920  0.792353 -0.008441  0.481077  1.000000

✅ DONE — Correlation + Full Analysis Ready!
